# AI System Design Case Studies

Analyze real-world AI systems from requirements to architecture to trade-offs.

## Overview

This notebook presents three detailed case studies of AI-powered systems. For each case study, we walk through:

1. **Requirements** — What the system needs to do
2. **Architecture** — How the system is designed
3. **Trade-offs** — What decisions were made and why
4. **Key Decisions** — Critical architectural choices

These case studies illustrate how the design principles from Module 19 apply in practice.

---
## Case Study 1: AI-Powered Search Engine

### Scenario

A media company wants to build an AI-powered search engine for their content library of 10 million articles. Users should be able to search using natural language and receive relevant, ranked results with AI-generated summaries.

### 1.1 Requirements

#### Business Context
- 10 million articles across multiple languages
- 500K monthly active users
- Current keyword search yields poor relevance (30% click-through rate)
- Goal: Improve to 60% click-through rate

#### Functional Requirements
- Natural language query understanding
- Semantic search (meaning, not just keywords)
- AI-generated snippet/summary for each result
- Multi-language support (10 languages)
- Filtering by date, category, author
- Spell correction and query suggestions

#### Non-Functional Requirements
- Latency (p95): < 500ms for search results
- Throughput: 10K queries/second peak
- Index freshness: New articles searchable within 5 minutes
- Uptime: 99.95%
- Cost: < $0.001 per query

### 1.2 Architecture

```mermaid
graph TB
    subgraph Frontend
        SearchUI[Search Interface]
        Autocomplete[Autocomplete]
    end

    subgraph API Layer
        Gateway[API Gateway]
        QueryParser[Query Parser]
    end

    subgraph AI Services
        EmbedService[Embedding Service]
        Reranker[Reranker Model]
        Summarizer[Summary Generator]
    end

    subgraph Search Infrastructure
        VectorIndex[(Vector Index)]
        BM25Index[(BM25 Index)]n        HybridMerger[Result Merger]
    end

    subgraph Data Pipeline
        Ingestion[Content Ingestion]
        Chunker[Document Chunker]
        EmbeddingPipeline[Embedding Pipeline]
    end

    SearchUI --> Gateway
    Autocomplete --> Gateway
    Gateway --> QueryParser
    QueryParser --> EmbedService
    QueryParser --> BM25Index
    EmbedService --> VectorIndex
    VectorIndex --> HybridMerger
    BM25Index --> HybridMerger
    HybridMerger --> Reranker
    Reranker --> Summarizer
    Ingestion --> Chunker
    Chunker --> EmbeddingPipeline
    EmbeddingPipeline --> VectorIndex
```

### 1.3 Key Decisions

#### Decision 1: Hybrid Search (BM25 + Vector)

| Approach | Relevance | Speed | Cost | Complexity |
|----------|-----------|-------|------|------------|
| BM25 only | Medium | Fast | Low | Low |
| Vector only | High | Medium | Medium | Medium |
| **Hybrid** | **High** | **Medium** | **Medium** | **Medium-High** |

**Rationale**: BM25 excels at exact keyword matching; vectors excel at semantic similarity. Combining them captures both.

#### Decision 2: Embedding Model Selection

| Model | Quality | Latency | Cost | Size |
|-------|---------|---------|------|------|
| OpenAI text-embedding-3-small | High | 50ms | $0.02/1M tokens | API |
| BGE-large-en | High | 30ms | Free (self-hosted) | 1.3GB |
| **GTE-base** | **High** | **20ms** | **Free** | **400MB** |

**Rationale**: GTE-base offers best quality-to-size ratio for self-hosting. Latency is critical for search.

#### Decision 3: Summary Generation

Use extractive summarization (no LLM) for speed and cost. Fall back to abstractive (LLM) for complex queries.

```python
# Pseudocode for search pipeline
def search(query: str, filters: dict) -> SearchResults:
    # Step 1: Parse and embed query
    parsed = parse_query(query)
    query_embedding = embed(parsed.text)
    
    # Step 2: Hybrid retrieval
    bm25_results = bm25_index.search(parsed.tokens, top_k=100)
    vector_results = vector_index.search(query_embedding, top_k=100)
    
    # Step 3: Merge and rerank
    merged = reciprocal_rank_fusion(bm25_results, vector_results)
    reranked = reranker.rerank(query, merged[:20])
    
    # Step 4: Generate summaries
    for result in reranked[:10]:
        result.summary = extractive_summary(result.passage, query)
    
    return SearchResults(results=reranked)
```

### 1.4 Trade-offs

| Dimension | Choice | Rationale |
|-----------|--------|-----------|
| **Accuracy vs Speed** | Hybrid search adds ~100ms but improves relevance by 40% | Click-through rate is the primary metric |
| **Cost vs Quality** | Self-hosted embeddings vs API | High volume makes API cost prohibitive |
| **Freshness vs Cost** | Real-time indexing vs batch | 5-minute freshness requirement |
| **Simplicity vs Performance** | Extractive summaries vs LLM | Speed and cost at scale |

### 1.5 Cost Estimation

```markdown
## Monthly Cost Estimate

| Component | Config | Cost |
|-----------|--------|------|
| Vector Index (Qdrant) | 3 nodes, 64GB RAM | $1,500 |
| BM25 Index (Elasticsearch) | 3 nodes | $1,200 |
| Embedding Service | 2x GPU (T4) | $800 |
| Reranker Service | 1x GPU (T4) | $400 |
| API Gateway + LB | Standard | $200 |
| CDN + Frontend | Cloudflare | $100 |
| Monitoring | Datadog | $300 |
| **Total** | | **$4,500/month** |
| **Cost per query** | 10M queries/month | **$0.00045** |
```

---
## Case Study 2: Real-Time Recommendation System

### Scenario

An e-commerce platform needs a real-time recommendation system that suggests products to users based on their browsing behavior, purchase history, and similar user patterns. The system must deliver personalized recommendations within 100ms.

### 2.1 Requirements

#### Business Context
- 5 million products, 20 million registered users
- 100K daily active users
- Current conversion rate: 2.1%
- Goal: Increase to 3.5% through better recommendations
- Recommendations appear on homepage, product pages, cart, and email

#### Functional Requirements
- "Recommended for you" section (personalized)
- "Customers also bought" (item-based collaborative)
- "Trending now" (popular items)
- "Recently viewed" (session-based)
- Real-time updates as user browses
- A/B testing framework for algorithms

#### Non-Functional Requirements
- Latency (p99): < 100ms for recommendation retrieval
- Throughput: 5K requests/second
- Freshness: Recommendations update within 30 seconds of user action
- Cold start: New users get recommendations within 3 clicks
- Uptime: 99.99%

### 2.2 Architecture

```mermaid
graph TB
    subgraph User Touchpoints
        Web[Web App]
        Mobile[Mobile App]
        Email[Email Service]
    end

    subgraph API Layer
        RecAPI[Recommendation API]
        ABTest[A/B Test Router]
    end

    subgraph Real-time Layer
        EventProcessor[Event Processor]
        UserTracker[User Session Tracker]
        RealTimeRanker[Real-time Ranker]
    end

    subgraph Batch Layer
        FeatureStore[(Feature Store)]
        SimilarityEngine[Similarity Engine]
        BatchPipeline[Batch Training Pipeline]
    end

    subgraph Storage
        UserProfiles[(User Profiles)]
        ItemIndex[(Item Index)]
        EventStore[(Event Store)]
    end

    Web --> RecAPI
    Mobile --> RecAPI
    Email --> RecAPI
    RecAPI --> ABTest
    ABTest --> RealTimeRanker
    RecAPI --> FeatureStore
    EventProcessor --> EventStore
    EventProcessor --> UserTracker
    UserTracker --> RealTimeRanker
    BatchPipeline --> FeatureStore
    BatchPipeline --> SimilarityEngine
    SimilarityEngine --> ItemIndex
```

### 2.3 Key Decisions

#### Decision 1: Two-Tier Architecture (Real-time + Batch)

**Rationale**: Recommendations need both long-term preferences (batch) and real-time context (streaming). Separating these allows independent scaling and updates.

```mermaid
flowchart LR
    A[User Request] --> B{Real-time Context?}
    B -->|Yes| C[Real-time Layer]
    B -->|No| D[Batch Features]
    C --> E[Combine Features]
    D --> E
    E --> F[Final Ranking]
```

#### Decision 2: Candidate Generation Strategy

| Strategy | Coverage | Latency | Personalization |
|----------|----------|---------|------------------|
| Collaborative filtering | High | Medium | High |
| Content-based | Medium | Low | Medium |
| popularity-based | Low | Very Low | None |
| **Two-stage: CF + re-ranking** | **High** | **Low** | **High** |

**Rationale**: Generate broad candidates with collaborative filtering (offline), then re-rank with real-time signals (online).

#### Decision 3: Feature Store Design

```markdown
## Feature Categories

| Category | Examples | Freshness | Storage |
|----------|----------|-----------|--------|
| User profile | Age, location, preferences | Daily | Redis |
| User behavior | Browse history, clicks | Real-time | Redis |
| Item features | Category, price, popularity | Hourly | Redis |
| Context | Time of day, device, location | Real-time | In-memory |
| Cross features | User-item affinity scores | Hourly | Redis |
```

### 2.4 Trade-offs

| Dimension | Choice | Rationale |
|-----------|--------|-----------|
| **Latency vs Freshness** | Pre-compute candidates, real-time re-rank | < 100ms requirement |
| **Cold Start vs Quality** | Content-based for new users, CF for established | Handle 30% new users |
| **Personalization vs Privacy** | Session-based tracking, no long-term storage | GDPR compliance |
| **A/B Testing vs Simplicity** | Built-in experimentation framework | Need to iterate on algorithms |

### 2.5 Latency Budget Breakdown

```markdown
## 100ms Latency Budget

| Step | Time | Notes |
|------|------|-------|
| API Gateway + Auth | 5ms | JWT validation |
| Feature Lookup | 10ms | Redis multi-get |
| Candidate Generation | 30ms | Pre-computed ANN search |
| Real-time Re-ranking | 20ms | Lightweight model |
| Response Serialization | 5ms | JSON encoding |
| Network + Buffer | 30ms | Safety margin |
| **Total** | **100ms** | |
```

---
## Case Study 3: Document Analysis Platform

### Scenario

A legal tech company needs a platform that analyzes legal documents (contracts, filings, agreements) to extract key clauses, identify risks, compare documents, and generate summaries. The system must handle documents up to 500 pages and deliver results within 30 seconds.

### 3.1 Requirements

#### Business Context
- Law firms and corporate legal teams
- Documents: contracts, NDAs, M&A filings, court documents
- Average document: 50 pages, 200 pages for complex filings
- Current process: Manual review takes 4-8 hours per document
- Goal: Reduce to 15-30 minutes with AI assistance

#### Functional Requirements
- Document upload and parsing (PDF, DOCX, scanned images)
- Key clause extraction (termination, liability, indemnification)
- Risk identification and scoring
- Document comparison (redline)
- Natural language summary generation
- Question answering over documents
- Export to standard formats

#### Non-Functional Requirements
- Processing time: < 30 seconds for 50-page document
- Accuracy: > 95% for clause extraction
- Security: SOC2 compliant, data encryption at rest and in transit
- Document size: Up to 500 pages
- Concurrent processing: 100 documents simultaneously
- Uptime: 99.9%

### 3.2 Architecture

```mermaid
graph TB
    subgraph Frontend
        DocUI[Document Upload UI]
        Viewer[Document Viewer]
        ChatUI[Q&A Interface]
    end

    subgraph API Layer
        Gateway[API Gateway]
        JobQueue[Job Queue]
    end

    subgraph Processing Pipeline
        Parser[Document Parser]
        OCR[OCR Service]
        Chunker[Smart Chunker]
        Classifier[Clause Classifier]
        Extractor[Entity Extractor]
        RiskScorer[Risk Scorer]
    end

    subgraph AI Services
        LLM[LLM Service]
        EmbedService[Embedding Service]
        CompareService[Document Comparison]
    end

    subgraph Storage
        DocStore[(Document Store)]
        VectorDB[(Vector DB)]
        MetadataDB[(Metadata DB)]
        ResultsCache[(Results Cache)]
    end

    DocUI --> Gateway
    Gateway --> JobQueue
    JobQueue --> Parser
    Parser --> OCR
    OCR --> Chunker
    Chunker --> Classifier
    Chunker --> Extractor
    Classifier --> RiskScorer
    RiskScorer --> LLM
    LLM --> DocStore
    Chunker --> EmbedService
    EmbedService --> VectorDB
    Viewer --> Gateway
    ChatUI --> Gateway
    ChatUI --> VectorDB
    ChatUI --> LLM
    CompareService --> DocStore
```

### 3.3 Key Decisions

#### Decision 1: Async Processing with Job Queue

**Rationale**: Document processing is compute-intensive (parsing, OCR, LLM calls). Synchronous processing would block the API and create poor user experience.

```mermaid
sequenceDiagram
    participant U as User
    participant API as API
    participant Q as Job Queue
    participant P as Processor
    participant S as Storage

    U->>API: Upload document
    API->>Q: Create job
    API-->>U: Job ID (immediate)
    Q->>P: Process document
    P->>S: Store parsed content
    P->>Q: Mark job complete
    U->>API: Check status / Get results
    API->>S: Retrieve results
    API-->>U: Document analysis
```

#### Decision 2: Chunking Strategy

Legal documents have complex structure. Smart chunking preserves context.

| Strategy | Context Preservation | Accuracy | Speed |
|----------|---------------------|----------|-------|
| Fixed-size chunks | Low | Medium | Fast |
| Paragraph-based | Medium | Medium | Fast |
| **Section-aware** | **High** | **High** | **Medium** |
| Semantic chunking | Very High | Very High | Slow |

**Rationale**: Legal documents follow predictable structure (sections, subsections, clauses). Section-aware chunking respects this structure while maintaining reasonable speed.

#### Decision 3: Model Strategy

```markdown
## Model Selection by Task

| Task | Model | Rationale |
|------|-------|-----------|
| OCR | Azure Form Recognizer | Best accuracy for legal docs |
| Clause Classification | Fine-tuned BERT | Domain-specific, fast |
| Entity Extraction | GPT-4 | Complex extraction, moderate volume |
| Risk Scoring | Custom model | Domain-specific rules + ML |
| Summary Generation | GPT-4 | High quality summaries |
| Q&A | GPT-4 + RAG | Context-aware answers |
```

### 3.4 Trade-offs

| Dimension | Choice | Rationale |
|-----------|--------|-----------|
| **Latency vs Accuracy** | Async processing, 30s target | Complex analysis needs time |
| **Cost vs Quality** | GPT-4 for critical tasks, BERT for routine | Balance quality and cost |
| **Security vs Convenience** | Strict access controls, no PII logging | Legal industry requirements |
| **Automation vs Human Review** | AI suggests, human approves | High-stakes decisions |

### 3.5 Security Architecture

```mermaid
graph TB
    User[User] --> Auth[Authentication]
    Auth --> AuthZ[Authorization]
    AuthZ --> Encryption[Encryption Layer]
    Encryption --> Processing[Processing]
    Processing --> AuditLog[Audit Logging]
    Processing --> DataLoss[Data Loss Prevention]
    
    subgraph Security Controls
        Auth
        AuthZ
        Encryption
        AuditLog
        DataLoss
    end

    subgraph Compliance
        SOC2[SOC2 Controls]
        GDPR[GDPR Controls]
        HIPAA[HIPAA if applicable]
    end
```

### 3.6 Cost Estimation

```markdown
## Monthly Cost Estimate (100K documents/month)

| Component | Config | Cost |
|-----------|--------|------|
| Document Parser (OCR) | Azure Form Recognizer | $2,000 |
| GPU Cluster (4x A10G) | Model serving | $3,200 |
| LLM API (GPT-4) | 100K docs × ~10 calls each | $15,000 |
| Vector DB (Pinecone) | Standard tier | $700 |
| Job Queue (SQS) | Standard | $100 |
| Storage (S3) | 5TB | $115 |
| Monitoring | Datadog | $500 |
| **Total** | | **$21,615/month** |
| **Cost per document** | | **$0.22** |
```

---
## Comparison Across Case Studies

### Architecture Patterns

| Pattern | Search Engine | Recommendations | Document Analysis |
|---------|---------------|-----------------|--------------------|
| Retrieval | Hybrid search | Two-stage | RAG |
| Processing | Synchronous | Real-time + Batch | Asynchronous |
| State | Stateless | Session-based | Job-based |
| Primary AI | Embeddings + Reranking | Collaborative filtering | LLM + Classification |
| Storage | Vector + Inverted index | Feature store + Cache | Document store + Vector |

### Key Lessons

1. **Search Engine**: Hybrid approaches often outperform single methods
2. **Recommendations**: Real-time and batch layers serve different purposes
3. **Document Analysis**: Async processing is essential for compute-intensive tasks
4. **All**: Cost estimation must account for both infrastructure and API costs
5. **All**: Security requirements drive architecture decisions significantly

### Common Anti-Patterns to Avoid

- **Over-engineering**: Don't build a feature store for a simple recommendation system
- **Under-estimating LLM costs**: API costs scale linearly with usage
- **Ignoring cold start**: New users/items need fallback strategies
- **Skipping monitoring**: You can't improve what you can't measure
- **Tight coupling**: Components should be replaceable independently

---
## Practice

For each case study, try to:

1. Identify one alternative architecture that could work
2. List three risks not mentioned in the analysis
3. Estimate how costs would change at 10x scale
4. Identify one security concern specific to each domain

Then create your own case study for a system you're familiar with, following the same structure.